In [1]:
import pandas as pd
from geopy.distance import great_circle
import numpy as np
df = pd.read_csv("../data/raw/fraudTrain.csv")
pd.set_option('display.max_columns', None)

In [2]:
def clean_data(df:pd.DataFrame):
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["date_of_birth"] = pd.to_datetime(df["dob"])
    df["merchant"] = df["merchant"].apply(lambda x:x.replace("fraud_", "") )
    
    df.drop(columns=["Unnamed: 0","dob"],inplace=True)
    df = df.sort_values(["cc_num", "trans_date_trans_time"]) # sort values in the start
    return df
    
    
def add_time(df:pd.DataFrame):
    df["week_day"] = df["trans_date_trans_time"].dt.day_name()
    df["hour"] = df["trans_date_trans_time"].dt.hour
    return df

    
def add_age(df:pd.DataFrame):
    df["age"] = df["trans_date_trans_time"].dt.year - df["date_of_birth"].dt.year
    bins = [0, 17, 24, 34, 49, 64, 100]
    labels = ["0-17", "18-24", "25-34","35-49", "50-64","65+"]
    df['age_group'] = pd.cut(df['age'],right=True, bins=bins, labels=labels, include_lowest=True)
    return df


def add_city_type(df: pd.DataFrame):
    bins = [0, 1000, 10000, 100000, 500000, 1000000, 10000000]
    labels = [
        "village",          # <1k
        "small town",       # 1k–10k
        "town",             # 10k–100k
        "small city",       # 100k–500k
        "medium city",      # 500k–1M
        "large city"        # 1M+
    ]
    df['city_type'] = pd.cut(
        df['city_pop'],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return df


def calc_distance(a_lat:float, a_long:float, b_lat:float, b_long:float):
    point_a = (a_lat, a_long)
    point_b = (b_lat, b_long)
    return great_circle(point_a, point_b).km
    


def add_distance(df:pd.DataFrame):
    df["distance_from_home"] = df.apply(lambda row: calc_distance(row["lat"], row["long"], row["merch_lat"], row["merch_long"]), axis=1)
    return df


def add_distance_bucket(df: pd.DataFrame):
    bins = [0, 5, 20, 100, 500, float("inf")]

    labels = [
        "Very Close (0–5 km)",
        "Nearby (5–20 km)",
        "City-Level (20–100 km)",
        "Far (100–500 km)",
        "Very Far (>500 km)"
    ]

    df["distance_group"] = pd.cut(
        df["distance_from_home"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    # Add sort column for Power BI
    df["distance_group_order"] = df["distance_group"].cat.codes

    return df


def add_amt_mean(df:pd.DataFrame):
    df["mean_amt"] = (
        df.groupby("cc_num")["amt"]
        .expanding()
        .mean()
        .shift()
        .reset_index(level=0, drop=True)
    )
    return df


    
def add_amt_std(df:pd.DataFrame):
    df["amt_std"] = (
        df.groupby("cc_num")["amt"]
        .expanding()
        .std()
        .shift()
        .reset_index(level=0, drop=True)
    )
    return df


def add_time_since(df:pd.DataFrame):
    df["time_since_prev_transaction"] = (df.groupby("cc_num")["trans_date_trans_time"].diff())
    return df

def add_prev_transaction_distance(df):
    
    # get previous merchant coordinates per user
    df["prev_merch_lat"] = df.groupby("cc_num")["merch_lat"].shift(1)
    df["prev_merch_long"] = df.groupby("cc_num")["merch_long"].shift(1)

    # compute distance between current and previous merchant location
    df["distance_from_prev"] = df.apply(
        lambda row: calc_distance(
            row["prev_merch_lat"],
            row["prev_merch_long"],
            row["merch_lat"],
            row["merch_long"]
        ) if not np.isnan(row["prev_merch_lat"]) else 0,
        axis=1
    )

    # cleanup
    df.drop(columns=["prev_merch_lat", "prev_merch_long"], inplace=True)

    return df

def add_speed(df:pd.DataFrame):
    df["time_since_prev_transaction"] = df["time_since_prev_transaction"].fillna(pd.Timedelta(seconds=0.001))
    df["time_since_prev_transaction_hours"] = df["time_since_prev_transaction"].dt.total_seconds() / 3600
    df["speed"] = df["distance_from_prev"] / df["time_since_prev_transaction_hours"]
    df["speed"] = df["speed"].replace([np.inf, -np.inf], 0)
    return df 
def add_speed_group(df: pd.DataFrame):
    # Keep inf as very large value instead of killing it
    df["speed"] = df["speed"].replace([np.inf, -np.inf], np.nan)

    # cap extreme values 
    df["speed_capped"] = df["speed"].clip(upper=1000)

    bins = [0, 5, 40, 120, 300, 800, float("inf")]

    labels = [
        "Very Low (0–5 km/h)",
        "Low (5–40 km/h)",
        "Moderate (40–120 km/h)",
        "High (120–300 km/h)",
        "Very High (300–800 km/h)",
        "Extreme (>800 km/h)"
    ]

    df["speed_group"] = pd.cut(
        df["speed_capped"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    # Sort order for Power BI
    df["speed_group_order"] = df["speed_group"].cat.codes

    return df

def add_amt_vs_mean_group(df: pd.DataFrame):
    # Avoid division issues
    df["amt_vs_avg"] = df["amt"] / df["mean_amt"]
    
    df.loc[df["mean_amt"] == 0, "amt_vs_avg"] = 0
    df["amt_vs_avg"] = df["amt_vs_avg"].fillna(0)
    
    bins = [0, 0.5, 0.9, 1.1, 2, 5, float("inf")]

    labels = [
        "Much Lower (<0.5x)",
        "Lower (0.5x–0.9x)",
        "Typical (0.9x–1.1x)",
        "Above Usual (1.1x–2x)",
        "High Spike (2x–5x)",
        "Extreme Spike (>5x)"
    ]

    df["amt_vs_avg_group"] = pd.cut(
        df["amt_vs_avg"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    # Create numeric sort order for Power BI
    df["amt_vs_avg_group_order"] = df["amt_vs_avg_group"].cat.codes

    return df
def flag_new_users(df:pd.DataFrame):
    df["is_new_user"] = df["mean_amt"].isna().astype(int)
    return df

def add_trans_date(df:pd.DataFrame):
    df["trans_date"] = df["trans_date_trans_time"].dt.normalize()
    return df
def add_z_score(df:pd.DataFrame):
    df["amt_std"] = df["amt_std"].replace(0, 1e-6)
    df["z_score"] = (df['amt'] - df["mean_amt"])/ df['amt_std']
    df['z_score'] = df['z_score'].fillna(0)
    return df

def fill_mean_std_nas(df):
    df["mean_amt"] = df["mean_amt"].fillna(0)
    df['amt_std'] = df['amt_std'].fillna(0)
    return df

In [ ]:


df = clean_data(df)
df = add_time(df)
df = add_age(df)
df = add_city_type(df)
df = add_distance(df)
df = add_distance_bucket(df)
df = add_amt_mean(df)
df = add_amt_std(df)
df = flag_new_users(df)
df = add_time_since(df)
df = add_prev_transaction_distance(df)
df = add_speed(df)
df = add_speed_group(df)
df = add_amt_vs_mean_group(df)
df = add_trans_date(df)
df = add_z_score(df)
df = fill_mean_std_nas(df)
df.to_csv("../data/processed/cleeaned_fraud_train.csv", index=False)

In [5]:
df.to_csv("../data/processed/cleeaned_fraud_train.csv", index=False)


In [4]:
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,trans_num,unix_time,merch_lat,merch_long,is_fraud,date_of_birth,week_day,hour,age,age_group,city_type,distance_from_home,distance_group,distance_group_order,mean_amt,amt_std,is_new_user,time_since_prev_transaction,distance_from_prev,time_since_prev_transaction_hours,speed,speed_capped,speed_group,speed_group_order,amt_vs_avg,amt_vs_avg_group,amt_vs_avg_group_order,trans_date,z_score
1017,2019-01-01 12:47:15,60416207185,"Jones, Sawayn and Romaguera",misc_net,7.27,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,98e3dcf98101146a577f85a34e58feec,1325422035,43.974711,-109.741904,0,1986-02-17,Tuesday,12,33,25-34,small town,127.606419,Far (100–500 km),3,0.000000,0.000000,1,0 days 00:00:00.001000,0.000000,2.777778e-07,0.000000,0.000000,Very Low (0–5 km/h),0,0.000000,Much Lower (<0.5x),0,2019-01-01,0.000000
2724,2019-01-02 08:44:57,60416207185,Berge LLC,gas_transport,52.94,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,498120fc45d277f7c88e3dba79c33865,1325493897,42.018766,-109.044172,0,1986-02-17,Wednesday,8,33,25-34,small town,110.309077,Far (100–500 km),3,7.270000,0.000000,0,0 days 19:57:42,224.769536,1.996167e+01,11.260059,11.260059,Low (5–40 km/h),1,7.281981,Extreme Spike (>5x),5,2019-01-02,0.000000
2726,2019-01-02 08:47:36,60416207185,Luettgen PLC,gas_transport,82.08,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,95f514bb993151347c7acdf8505c3d62,1325494056,42.961335,-109.157564,0,1986-02-17,Wednesday,8,33,25-34,small town,21.787292,City-Level (20–100 km),2,30.105000,32.293567,0,0 days 00:02:39,105.220587,4.416667e-02,2382.352919,1000.000000,Extreme (>800 km/h),5,2.726457,High Spike (2x–5x),4,2019-01-02,1.609454
2882,2019-01-02 12:38:14,60416207185,Daugherty LLC,kids_pets,34.79,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,4f0c1a14e0aa7eb56a490780ef9268c5,1325507894,42.228227,-108.747683,0,1986-02-17,Wednesday,12,33,25-34,small town,87.204338,City-Level (20–100 km),2,47.430000,37.708144,0,0 days 03:50:38,88.152407,3.843889e+00,22.933131,22.933131,Low (5–40 km/h),1,0.733502,Lower (0.5x–0.9x),1,2019-01-02,-0.335206
2907,2019-01-02 13:10:46,60416207185,Beier and Sons,home,27.18,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,3b2ebd3af508afba959640893e1e82bc,1325509846,43.321745,-108.091143,0,1986-02-17,Wednesday,13,33,25-34,small town,74.213070,City-Level (20–100 km),2,44.270000,31.430534,0,0 days 00:32:32,132.876960,5.422222e-01,245.059968,245.059968,High (120–300 km/h),3,0.613960,Lower (0.5x–0.9x),1,2019-01-02,-0.543739
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294934,2020-06-20 21:04:59,4992346398065154184,"Berge, Kautzer and Harris",personal_care,60.47,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,ad7dfdf0aaa36cd7985dd1f35ca0e2fc,1371762299,40.475395,-89.076105,0,1956-01-09,Saturday,21,64,50-64,village,78.492673,City-Level (20–100 km),2,67.802931,157.357527,0,0 days 08:32:20,72.134080,8.538889e+00,8.447713,8.447713,Low (5–40 km/h),1,0.891849,Lower (0.5x–0.9x),1,2020-06-20,-0.046600
1295369,2020-06-21 00:41:01,4992346398065154184,Bernhard Inc,gas_transport,74.29,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,6d427d735c9f9b2fd480f2c24b6525de,1371775261,40.743634,-89.553379,0,1956-01-09,Sunday,0,64,50-64,village,55.400846,City-Level (20–100 km),2,67.799363,157.319300,0,0 days 03:36:02,50.128303,3.600556e+00,13.922380,13.922380,Low (5–40 km/h),1,1.095733,Typical (0.9x–1.1x),2,2020-06-21,0.041258
1295587,2020-06-21 0

In [5]:
# 7. Velocity Features

# For each transaction:

# number of transactions in last:
# 1 hour
# 24 hours

# Transaction Behavior per User


# assign day/night
# assign weekday/weekend

# distance from previous transaction

In [6]:
df[df["is_fraud"] == 1]["speed"].mean()

np.float64(742.93178958857)

In [7]:
df[df["is_fraud"] == 0]["speed"].mean()

np.float64(185.79334979702512)

In [8]:
# #Z-scores: Standardizing transaction features (e.g., transaction amount, frequency) to detect how
# # far a given data point is from the mean can help identify extreme or outlier values that may indicate
# # fraudulent activity.

# # Moving Averages: Averages over a defined time window (e.g., the last 30 days of transaction
# # amounts) help smooth out fluctuations and highlight sudden changes in behavior.

# # Deviation from User Norms: Comparing a user’s current activity with their historical averages
# # (e.g., average spend, average transaction frequency) can help highlight anomalous activity that
# # could suggest fraud.

# # Moving Averages: Averages over a defined time window (e.g., the last 30 days of transaction
# # amounts) help smooth out fluctuations and highlight sudden changes in behavior.Calculating moving averages (e.g., 7-day or 30-day) for features like
# # transaction amount or frequency can help smooth out short-term fluctuations and highlight longterm trends. For example, a sudden increase in the moving average of transaction amounts can
# # indicate potential fraudulent behavior

# Transaction Volatility: The volatility or standard deviation of transaction amounts, frequencies, or
# other features over a defined period can provide valuable insight into unusual behavior. A user
# with very high volatility in transaction amounts may be engaged in fraud or other suspicious
# activities.

# Aggregated Transaction Sums: Summing transactions over various time windows (e.g., daily,
# weekly, or monthly) can help to detect abnormal behavior. For instance, if a user has a significant
# number of transactions within a short period, it could suggest fraudulent activity or a compromised
# account.

# Mahalanobis Distance: This is a distance measure that accounts for the correlation between
# features, which can be more effective in detecting anomalies in multivariate datasets. It is useful
# when dealing with high-dimensional fraud detection problems, where multiple features interact in
# complex ways

# Sliding Window Features: A sliding window approach involves calculating features over a rolling
# time window. For example, analyzing a user’s transaction behavior over the last 10 minutes, 1
# hour, or 24 hours can help identify rapidly changing patterns indicative of fraud. These features
# are especially useful for real-time fraud detection systems.
# Seasonality Patterns: Many financial behaviors exhibit seasonal patterns (e.g., higher spending
# during holidays). Identifying seasonal trends within user behavior (e.g., spending patterns,
# frequency of transactions) can help detect anomalies when behavior deviates from typical seasonal
# patterns.


# Filter methods evaluate each feature independently of the machine learning model and select
# features based on statistical tests or their correlation with the target variable (fraud vs. non-fraud).
# These methods are computationally efficient and work well when dealing with large datasets.


# Chi-Square Test: The chi-square test assesses the independence between a feature and the target
# variable. Features that are highly dependent on the target variable (i.e., show significant
# association) are retained, while those with low dependency are discarded. This is particularly
# useful for categorical features

# Recursive Feature Elimination (RFE): RFE is a popular method that recursively removes the least
# important features based on the performance of a chosen model. The algorithm starts by training
# the model with all features, then removes the least significant feature and retrains the model. This
# process continues until the desired number of features is reached. RFE works well with models
# like decision trees, logistic regression, and support vector machines.

# Z-Score or IQR-Based Methods: Statistical methods such as the Z-score (which identifies values
# that are many standard deviations away from the mean) or the Interquartile Range (IQR, which
# captures values outside 1.5 times the IQR from the 25th and 75th percentiles) can be used to
# identify outliers.
# Oversampling: The minority class (fraudulent transactions) can be oversampled by duplicating
# instances or generating synthetic data using techniques like SMOTE (Synthetic Minority Oversampling Technique).

# Anomaly Detection Models: Given that fraud is relatively rare, treating fraud as an anomaly or
# outlier detection problem can be effective. Algorithms such as Isolation Forest, One-Class SVM,
# or autoencoders can be used to identify rare events that deviate significantly from the norm.
# Evaluation Metrics: Using appropriate evaluation metrics is crucial when dealing with imbalanced
# datasets. Instead of relying on accuracy (which can be misleading in imbalanced scenarios),
# metrics such as Precision, Recall, F1-Score, and Area Under the Precision-Recall Curve (AUCPR) should be used to assess model performance.

# Transaction Sequence Analysis: Real-time fraud detection systems can analyze the sequence of
# transactions to identify patterns that deviate from typical sequences. For example, multiple small
# transactions leading to a larger one (often referred to as "smurfing") can indicate fraudulent intent.


In [9]:
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,trans_num,unix_time,merch_lat,merch_long,is_fraud,date_of_birth,week_day,hour,age,age_group,city_group,distance,distance_group,mean_amt,amt_std,time_since_prev_transaction,distance_from_prev,time_since_prev_transaction_hours,speed,amt_vs_avg,amt_vs_avg_group,trans_date
1017,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.27,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,98e3dcf98101146a577f85a34e58feec,1325422035,43.974711,-109.741904,0,1986-02-17,Tuesday,12,33,25-34,small_town,127.606419,plane,56.023366,122.632635,0 days 00:05:00,0.000000,0.083333,0.000000,0.129767,very_low,2019-01-01
2724,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.94,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,498120fc45d277f7c88e3dba79c33865,1325493897,42.018766,-109.044172,0,1986-02-17,Wednesday,8,33,25-34,small_town,110.309077,plane,56.023366,122.632635,0 days 19:57:42,224.769536,19.961667,11.260059,0.944963,low,2019-01-02
2726,2019-01-02 08:47:36,60416207185,fraud_Luettgen PLC,gas_transport,82.08,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,95f514bb993151347c7acdf8505c3d62,1325494056,42.961335,-109.157564,0,1986-02-17,Wednesday,8,33,25-34,small_town,21.787292,car,56.023366,122.632635,0 days 00:02:39,105.220587,0.044167,2382.352919,1.465103,normal,2019-01-02
2882,2019-01-02 12:38:14,60416207185,fraud_Daugherty LLC,kids_pets,34.79,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,4f0c1a14e0aa7eb56a490780ef9268c5,1325507894,42.228227,-108.747683,0,1986-02-17,Wednesday,12,33,25-34,small_town,87.204338,car,56.023366,122.632635,0 days 03:50:38,88.152407,3.843889,22.933131,0.620991,low,2019-01-02
2907,2019-01-02 13:10:46,60416207185,fraud_Beier and Sons,home,27.18,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,3b2ebd3af508afba959640893e1e82bc,1325509846,43.321745,-108.091143,0,1986-02-17,Wednesday,13,33,25-34,small_town,74.213070,car,56.023366,122.632635,0 days 00:32:32,132.876960,0.542222,245.059968,0.485155,very_low,2019-01-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294934,2020-06-20 21:04:59,4992346398065154184,"fraud_Berge, Kautzer and Harris",personal_care,60.47,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,ad7dfdf0aaa36cd7985dd1f35ca0e2fc,1371762299,40.475395,-89.076105,0,1956-01-09,Saturday,21,64,50-64,rural,78.492673,car,67.843832,157.223610,0 days 08:32:20,72.134080,8.538889,8.447713,0.891312,low,2020-06-20
1295369,2020-06-21 00:41:01,4992346398065154184,fraud_Bernhard Inc,gas_transport,74.29,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,6d427d735c9f9b2fd480f2c24b6525de,1371775261,40.743634,-89.553379,0,1956-01-09,Sunday,0,64,50-64,rural,55.400846,car,67.843832,157.223610,0 days 03:36:02,50.128303,3.600556,13.922380,1.095015,normal,2020-06-21
1295587,2020-06-21 02:47:59,4992346398065154184,"fraud_Reichert, Rowe and Mraz",shopping_net,246.56,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,9814049bcc69fb31d81f4a907f2fe255,1371782879,40.215418,-88.682562,0,1956-01-09,Sunday,2,64,50-64,rural,115.674563,plane,67.843832,157.223610,0 days 02:06:58,94.204046,2.116111,44.517533,3.634229,high,2020-06-21
1296206,2020-06-21 08:04:28,4992346398065154184,fraud_Jewess LLC,shopping_pos,2.62,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,ae39b91cd2b4897ddbbf6bf63d3e7b03,1371801868,40.762861,-88.744967,0,1956-01-09,Sunday,8,64,50